In [19]:
from pathlib import Path
import json
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

# If the notebook is running inside notebooks/, use:
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPAIR_DIR = DATA_DIR / "repaired"
AUDIT_DIR = DATA_DIR / "audit"

REPAIR_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed directory exists:", PROCESSED_DIR.exists())
print("Repair directory:", REPAIR_DIR)

Project root: c:\Users\Nihan\.vscode\pitwall
Processed directory exists: True
Repair directory: c:\Users\Nihan\.vscode\pitwall\data\repaired


In [20]:
EXPECTED_FILES = [
    "lap_performance.parquet",
    "race_state.parquet",
    "traffic_features.parquet",
    "fuel_features.parquet",
    "tyre_features.parquet",
]

# Search for the known datasets without modifying anything.
found_files = {}

for filename in EXPECTED_FILES:
    matches = list(PROJECT_ROOT.rglob(filename))
    matches = [
        p for p in matches
        if ".git" not in p.parts
        and "venv" not in p.parts
        and ".venv" not in p.parts
    ]

    if matches:
        found_files[filename] = matches[0]
        print(f"{filename}: {matches[0]}")
    else:
        print(f"MISSING: {filename}")

def load_parquet(filename):
    path = found_files.get(filename)
    if path is None:
        raise FileNotFoundError(
            f"{filename} was not found. Check the repository's data paths."
        )
    return pd.read_parquet(path)

datasets = {
    name: load_parquet(name)
    for name in found_files
}

for name, df in datasets.items():
    print(f"\n{'=' * 70}")
    print(name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

lap_performance.parquet: c:\Users\Nihan\.vscode\pitwall\data\processed\9558\lap_performance.parquet
race_state.parquet: c:\Users\Nihan\.vscode\pitwall\data\repaired\race_state.parquet
traffic_features.parquet: c:\Users\Nihan\.vscode\pitwall\data\repaired\traffic_features.parquet
fuel_features.parquet: c:\Users\Nihan\.vscode\pitwall\data\processed\9558\fuel_features.parquet
tyre_features.parquet: c:\Users\Nihan\.vscode\pitwall\data\processed\9558\tyre_features.parquet

lap_performance.parquet
Shape: (962, 19)
Columns: ['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age']
 meeting_key  session_key  driver_number  lap_number                       date_start  duration_sector_1  duration_sector_2  duration_sector_3  i1_speed  i2_speed  i

In [21]:
KEY_CANDIDATES = [
    ("driver_number", "lap_number"),
    ("driver_number_x", "lap_number"),
    ("driver_number", "lap"),
    ("driver_number_x", "lap"),
]

def identify_key(df):
    for cols in KEY_CANDIDATES:
        if all(c in df.columns for c in cols):
            return list(cols)
    return None

key_audit = []

for name, df in datasets.items():
    key = identify_key(df)

    if key is None:
        key_audit.append({
            "dataset": name,
            "rows": len(df),
            "key": None,
            "duplicate_rows": None,
            "missing_key_rows": None,
            "unique_keys": None,
        })
        print(f"\n{name}: no recognized key. Inspect columns manually.")
        continue

    duplicate_mask = df.duplicated(key, keep=False)
    missing_mask = df[key].isna().any(axis=1)

    key_audit.append({
        "dataset": name,
        "rows": len(df),
        "key": tuple(key),
        "duplicate_rows": int(duplicate_mask.sum()),
        "missing_key_rows": int(missing_mask.sum()),
        "unique_keys": int(df[key].drop_duplicates().shape[0]),
    })

    print(f"\n{name}")
    print("Key:", key)
    print("Rows:", len(df))
    print("Unique keys:", df[key].drop_duplicates().shape[0])
    print("Duplicate rows:", duplicate_mask.sum())
    print("Rows with missing keys:", missing_mask.sum())

    if duplicate_mask.any():
        print("\nDuplicate examples:")
        print(
            df.loc[duplicate_mask]
              .sort_values(key)
              .head(20)
              .to_string(index=False)
        )

key_audit_df = pd.DataFrame(key_audit)
key_audit_df


lap_performance.parquet
Key: ['driver_number', 'lap_number']
Rows: 962
Unique keys: 962
Duplicate rows: 0
Rows with missing keys: 0

race_state.parquet
Key: ['driver_number', 'lap_number']
Rows: 962
Unique keys: 962
Duplicate rows: 0
Rows with missing keys: 0

traffic_features.parquet
Key: ['driver_number', 'lap_number']
Rows: 961
Unique keys: 961
Duplicate rows: 0
Rows with missing keys: 0

fuel_features.parquet
Key: ['driver_number', 'lap_number']
Rows: 692
Unique keys: 692
Duplicate rows: 0
Rows with missing keys: 0

tyre_features.parquet
Key: ['driver_number', 'lap_number']
Rows: 650
Unique keys: 650
Duplicate rows: 0
Rows with missing keys: 0


,dataset,rows,key,duplicate_rows,missing_key_rows,unique_keys
0,lap_performance.parquet,962,"(driver_number, lap_number)",0,0,962
1,race_state.parquet,962,"(driver_number, lap_number)",0,0,962
2,traffic_features.parquet,961,"(driver_number, lap_number)",0,0,961
3,fuel_features.parquet,692,"(driver_number, lap_number)",0,0,692
4,tyre_features.parquet,650,"(driver_number, lap_number)",0,0,650


In [22]:
def remove_exact_duplicates_only(df, key, dataset_name):
    if df[key].isna().any(axis=1).any():
        raise ValueError(
            f"{dataset_name}: missing values in key {key}. "
            "Resolve these before deduplication."
        )

    duplicated = df[df.duplicated(key, keep=False)].copy()

    if duplicated.empty:
        print(f"{dataset_name}: keys already unique.")
        return df.copy()

    conflicting_keys = []

    for _, group in duplicated.groupby(key, dropna=False):
        # If all full rows are identical, dropping repeated copies is safe.
        if len(group.drop_duplicates()) > 1:
            conflicting_keys.append(group)

    if conflicting_keys:
        conflict_df = pd.concat(conflicting_keys).sort_values(key)
        conflict_path = AUDIT_DIR / f"{dataset_name}_conflicting_keys.csv"
        conflict_df.to_csv(conflict_path, index=False)

        raise ValueError(
            f"{dataset_name}: conflicting records share the same key. "
            f"Review {conflict_path}. No rows were discarded."
        )

    result = df.drop_duplicates().copy()

    if result.duplicated(key).any():
        raise AssertionError(
            f"{dataset_name}: duplicate keys remain after exact deduplication."
        )

    print(
        f"{dataset_name}: removed "
        f"{len(df) - len(result)} exact duplicate row(s)."
    )
    return result

In [23]:
lap_df = datasets["lap_performance.parquet"].copy()
fuel_df = datasets["fuel_features.parquet"].copy()

lap_key = identify_key(lap_df)
fuel_key = identify_key(fuel_df)

if lap_key is None or fuel_key is None:
    raise ValueError("Inspect the key columns before joining fuel features.")

print("Lap performance rows:", len(lap_df))
print("Fuel feature rows:", len(fuel_df))

print("\nFuel columns:")
print(fuel_df.columns.tolist())

print("\nFuel null counts:")
print(fuel_df.isna().sum().sort_values(ascending=False))

lap_keys = lap_df[lap_key].drop_duplicates()
fuel_keys = fuel_df[fuel_key].drop_duplicates()

# Normalize key column names before comparing.
lap_keys = lap_keys.rename(
    columns=dict(zip(lap_key, ["driver_number", "lap_number"]))
)
fuel_keys = fuel_keys.rename(
    columns=dict(zip(fuel_key, ["driver_number", "lap_number"]))
)

coverage = lap_keys.merge(
    fuel_keys,
    on=["driver_number", "lap_number"],
    how="left",
    indicator=True,
    validate="one_to_one"
)

print("\nCoverage:")
print(coverage["_merge"].value_counts())

print("\nMissing fuel-feature keys:")
print(
    coverage.loc[coverage["_merge"] == "left_only",
                 ["driver_number", "lap_number"]]
            .sort_values(["driver_number", "lap_number"])
            .to_string(index=False)
)

Lap performance rows: 962
Fuel feature rows: 692

Fuel columns:
['driver_number', 'lap_number', 'fuel_effect']

Fuel null counts:
driver_number    0
lap_number       0
fuel_effect      0
dtype: int64

Coverage:
_merge
both          692
left_only     270
right_only      0
Name: count, dtype: int64

Missing fuel-feature keys:
 driver_number  lap_number
             1          27
             1          28
             1          29
             1          30
             1          31
             1          32
             1          33
             1          34
             1          35
             1          36
             1          37
             1          38
             1          39
             2          27
             2          28
             2          29
             2          30
             2          31
             2          32
             2          33
             2          34
             2          35
             2          36
             2          37

In [24]:
def build_race_progress_features(laps):
    df = laps.copy()

    key = identify_key(df)
    if key is None:
        raise ValueError("No recognized driver-lap key.")

    driver_col, lap_col = key

    df[driver_col] = df[driver_col].astype("string")
    df[lap_col] = pd.to_numeric(df[lap_col], errors="coerce")

    if df[[driver_col, lap_col]].isna().any(axis=1).any():
        raise ValueError("Missing driver or lap number in lap performance data.")

    if df.duplicated([driver_col, lap_col]).any():
        raise ValueError(
            "lap_performance has duplicate driver-lap keys. "
            "Repair the source before creating features."
        )

    df = df.sort_values([driver_col, lap_col]).copy()

    # Number of laps observed for each driver in this dataset.
    # This is NOT necessarily the official race distance for retirees.
    observed_max_lap = df.groupby(driver_col)[lap_col].transform("max")

    denominator = (observed_max_lap - 1).replace(0, np.nan)

    df["race_progress_proxy"] = (
        (df[lap_col] - 1) / denominator
    ).fillna(0.0).clip(0.0, 1.0)

    df["laps_observed_remaining_proxy"] = (
        observed_max_lap - df[lap_col]
    ).clip(lower=0)

    df["fuel_mass_kg"] = np.nan
    df["fuel_effect_seconds"] = np.nan

    return df[
        [
            driver_col,
            lap_col,
            "race_progress_proxy",
            "laps_observed_remaining_proxy",
            "fuel_mass_kg",
            "fuel_effect_seconds",
        ]
    ].rename(columns={
        driver_col: "driver_number",
        lap_col: "lap_number",
    })

fuel_proxy = build_race_progress_features(lap_df)

fuel_proxy.to_parquet(
    REPAIR_DIR / "fuel_features_proxy.parquet",
    index=False
)

fuel_proxy.head()

,driver_number,lap_number,race_progress_proxy,laps_observed_remaining_proxy,fuel_mass_kg,fuel_effect_seconds
0,1,1,0.000000,51,NaN,NaN
1,1,2,0.019608,50,NaN,NaN
2,1,3,0.039216,49,NaN,NaN
3,1,4,0.058824,48,NaN,NaN
4,1,5,0.078431,47,NaN,NaN


In [25]:
race_state_repaired = race_state.copy()

if "grip_condition" in race_state_repaired.columns:
    print(
        "Grip values before:",
        race_state_repaired["grip_condition"].value_counts(dropna=False).to_dict()
    )

    # Apply only if NORMAL is the documented pre-intervention default
    # in your race-control feature definition.
    race_state_repaired["grip_condition"] = (
        race_state_repaired["grip_condition"].fillna("NORMAL")
    )

if "date" in race_state_repaired.columns:
    # Do not manufacture dates. If this column is entirely null and is not
    # needed downstream, remove it from the repaired feature export.
    if race_state_repaired["date"].isna().all():
        race_state_repaired = race_state_repaired.drop(columns=["date"])

race_state_repaired.to_parquet(
    REPAIR_DIR / "race_state.parquet",
    index=False
)

print("Remaining nulls:")
print(race_state_repaired.isna().sum().sort_values(ascending=False).head(15))

Grip values before: {'NORMAL': 585, 'LOW': 377}
Remaining nulls:
lap_duration           3
compound               2
tyre_age               2
stint_number           2
position               1
date_start             1
weather_age_seconds    1
pressure               1
session_key            1
meeting_key            1
wind_speed             1
track_temperature      1
rainfall               1
humidity               1
blue_flag              1
dtype: int64


In [28]:
INVALID_PACE_FEATURES = [
    "position",
    "traffic_ahead",
]

def remove_invalid_pace_features(df, columns=INVALID_PACE_FEATURES):
    found = [c for c in columns if c in df.columns]
    cleaned = df.drop(columns=found).copy()

    print("Removed from pace predictors:", found)
    print("Remaining columns:", cleaned.columns.tolist())

    return cleaned
print("Cell 9 executed successfully.")

print("Function available:",
      callable(remove_invalid_pace_features))

Cell 9 executed successfully.
Function available: True


In [29]:
def merge_driver_lap_features(left, right, name="feature_table"):
    left = left.copy()
    right = right.copy()

    left_key = identify_key(left)
    right_key = identify_key(right)

    if left_key is None or right_key is None:
        raise ValueError(
            f"{name}: could not identify join keys. "
            "Inspect driver_number and lap_number column names."
        )

    # Standardize key names in the right table.
    right = right.rename(columns={
        right_key[0]: left_key[0],
        right_key[1]: left_key[1],
    })

    key = left_key

    if left[key].isna().any(axis=1).any():
        raise ValueError("Left table has missing join keys.")

    if right[key].isna().any(axis=1).any():
        raise ValueError(f"{name} has missing join keys.")

    if left.duplicated(key).any():
        raise ValueError("Left table has duplicate driver-lap keys.")

    if right.duplicated(key).any():
        raise ValueError(f"{name} has duplicate driver-lap keys.")

    overlap = (set(left.columns) & set(right.columns)) - set(key)
    if overlap:
        raise ValueError(
            f"{name}: overlapping non-key columns {sorted(overlap)}. "
            "Rename them explicitly before merging."
        )

    merged = left.merge(
        right,
        on=key,
        how="left",
        validate="one_to_one",
        indicator=f"_{name}_merge",
    )

    print(f"\n{name}")
    print("Rows before:", len(left))
    print("Rows after:", len(merged))
    print(merged[f"_{name}_merge"].value_counts(dropna=False))

    if len(merged) != len(left):
        raise AssertionError("Merge unexpectedly changed the row count.")

    return merged.drop(columns=f"_{name}_merge")

print("Cell 10 executed successfully.")
print("Function available:",
      callable(merge_driver_lap_features))

Cell 10 executed successfully.
Function available: True


In [30]:
def audit_feature_availability(
    df,
    decision_time_col,
    feature_time_columns
):
    if decision_time_col not in df.columns:
        raise KeyError(f"Missing decision-time column: {decision_time_col}")

    result = []

    decision_time = pd.to_datetime(
        df[decision_time_col], utc=True, errors="coerce"
    )

    for feature, time_col in feature_time_columns.items():
        if time_col not in df.columns:
            result.append({
                "feature": feature,
                "timestamp_column": time_col,
                "checked_rows": 0,
                "future_rows": None,
                "missing_timestamps": None,
                "status": "COLUMN_MISSING",
            })
            continue

        feature_time = pd.to_datetime(
            df[time_col], utc=True, errors="coerce"
        )

        comparable = decision_time.notna() & feature_time.notna()
        future = comparable & (feature_time > decision_time)

        result.append({
            "feature": feature,
            "timestamp_column": time_col,
            "checked_rows": int(comparable.sum()),
            "future_rows": int(future.sum()),
            "missing_timestamps": int(
                (decision_time.notna() & feature_time.isna()).sum()
            ),
            "status": "FAIL" if future.any() else "REVIEW",
        })

        if future.any():
            print(f"\nFuture timestamps detected for {feature}:")
            display(
                df.loc[future, [decision_time_col, time_col]].head(10)
            )

    return pd.DataFrame(result)

In [31]:
def add_lagged_track_evolution(
    df,
    lap_col="lap_number",
    pace_col="lap_duration",
    driver_col="driver_number",
):
    required = [lap_col, pace_col, driver_col]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise KeyError(f"Missing columns: {missing}")

    out = df.copy()

    # Use only completed laps, and compute the field median by lap.
    field_median = (
        out.groupby(lap_col)[pace_col]
           .median()
           .sort_index()
    )

    # At lap N, use the field median from the latest available earlier lap.
    lagged_median = field_median.shift(1)

    out["field_median_previous_lap"] = out[lap_col].map(lagged_median)

    # Relative pace is useful for separating a driver's pace from broad
    # changes affecting the whole field. Negative means faster than median.
    out["pace_relative_to_previous_field_median"] = (
        out[pace_col] - out["field_median_previous_lap"]
    )

    return out

In [32]:
def validate_driver_lap_table(df, name):
    key = identify_key(df)

    assert key is not None, (
        f"{name}: driver-lap key not identified. "
        f"Columns are {df.columns.tolist()}"
    )

    assert not df[key].isna().any(axis=1).any(), (
        f"{name}: missing driver-lap keys."
    )

    assert not df.duplicated(key).any(), (
        f"{name}: duplicate driver-lap keys."
    )

    return {
        "dataset": name,
        "rows": len(df),
        "unique_keys": df[key].drop_duplicates().shape[0],
        "status": "PASS",
    }


validation_results = []

for name, df in datasets.items():
    try:
        validation_results.append(validate_driver_lap_table(df, name))
    except AssertionError as exc:
        validation_results.append({
            "dataset": name,
            "rows": len(df),
            "unique_keys": None,
            "status": f"FAIL: {exc}",
        })

display(pd.DataFrame(validation_results))

,dataset,rows,unique_keys,status
0,lap_performance.parquet,962,962,PASS
1,race_state.parquet,962,962,PASS
2,traffic_features.parquet,961,961,PASS
3,fuel_features.parquet,692,692,PASS
4,tyre_features.parquet,650,650,PASS


In [35]:
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project": "PITWALL",
    "purpose": "Controlled data integrity repairs",
    "original_data_modified": False,
    "repair_directory": str(REPAIR_DIR),
    "notes": [
        "Original processed datasets were not overwritten.",
        "Fuel progress proxy is not physical fuel mass.",
        "Grip defaults must be supported by documented feature semantics.",
        "Feature availability must be validated separately from join integrity.",
        "Historical compound medians are not causal tyre performance estimates.",
    ],
}

manifest_path = AUDIT_DIR / "repair_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Manifest saved:", manifest_path)

Manifest saved: c:\Users\Nihan\.vscode\pitwall\data\audit\repair_manifest.json
